# `epibyhand`: a hands-on tutorial

**Classical epidemiological measures that show their work.**

Most statistical software gives you the answer. In a methods course the answer
is the least interesting part of the calculation — a student who can produce
`2.14` without being able to say where it came from has learned nothing that
will survive the exam.

Every function in `epibyhand` returns the answer *together with the reasoning
that produced it*: each intermediate quantity, the formula in symbols, and the
formula with the observed numbers substituted in.

### What this tutorial covers

| Part | Topic |
|---|---|
| 1 | Building a 2 x 2 table |
| 2 | Risk ratio and risk difference |
| 3 | Odds ratio, and when it approximates the risk ratio |
| 4 | Controlling how much detail you see |
| 5 | Checking a hand calculation with `check_work()` |
| 6 | Attributable fractions |
| 7 | Confounding and stratified analysis |
| 8 | Homogeneity and effect modification |
| 9 | Building problem sets |
| — | **Eight exercises with worked answers** |

### Scope

`epibyhand` covers **methods a student can compute by hand on paper**. That
boundary is deliberate: it is why there is no regression modelling here. Once
an estimate comes from an iterative fit there is no hand calculation left to
check, and a printed "derivation" would be decoration rather than instruction.

---
## Setup

**This notebook needs an R runtime.** In Colab go to
**Runtime → Change runtime type → Runtime type: R**, then run the cell below.

(If the menu has no R option, open a fresh R notebook from
`https://colab.research.google.com/#create=true&language=r` and paste this
notebook's cells into it.)

In [1]:
# Install from CRAN. Takes about 30 seconds; there is nothing to compile.
if (!requireNamespace("epibyhand", quietly = TRUE)) {
  install.packages("epibyhand")
}

library(epibyhand)
packageVersion("epibyhand")

[1] ‘0.1.0’


---
## Part 1 — The 2 x 2 table

Everything starts here. `epi2x2()` takes the four cell counts in the standard
epidemiological orientation:

|  | Case | Non-case |
|---|---|---|
| **Exposed** | a | b |
| **Unexposed** | c | d |

So the argument order is **a, b, c, d** — reading across the top row, then
across the bottom row.

### Our first dataset

A church picnic is followed by an outbreak of gastroenteritis. Investigators
interview all 120 attendees and ask whether they ate the potato salad:

* Of the 70 who **ate** the potato salad, **54 became ill**.
* Of the 50 who **did not**, **8 became ill**.

In [2]:
picnic <- epi2x2(54, 16, 8, 42,
                 exposure = c("Ate potato salad", "Did not eat"),
                 outcome  = c("Ill", "Well"))
picnic

                  Ill  Well  Total
Ate potato salad   54    16     70
Did not eat         8    42     50
Total              62    58    120


The `exposure` and `outcome` arguments only change the labels — they do not
change any arithmetic. But labelling the table honestly is worth the extra
few characters, because it is what stops you from reading the output of a
case-control study as though it were a cohort.

**A note on orientation.** Getting a, b, c, d in the wrong order is the single
most common source of a wrong answer in this whole tutorial. If your odds
ratio comes out as the reciprocal of what you expected, you have almost
certainly swapped the rows or the columns.

---
## Part 2 — Risk ratio and risk difference

This is a **cohort**: we followed a defined group forward and counted who fell
ill. That means *risk is estimable* — we can divide cases by the number of
people at risk — and the risk ratio is available to us.

In [3]:
risk_ratio(picnic)

── Risk ratio ──────────────────────────────────────────────────────────────

Data
                    Ill  Well  Total
  Ate potato salad   54    16     70
  Did not eat         8    42     50
  Total              62    58    120

Step 1  Risk of Ill among the ate potato salad
        R1 = a / (a + b)
           = 54 / (54 + 16)
           = 0.7714

Step 2  Risk of Ill among the did not eat
        R0 = c / (c + d)
           = 8 / (8 + 42)
           = 0.16

Step 3  Risk ratio
        RR = R1 / R0
           = 0.7714 / 0.16
           = 4.8214

Step 4  Standard error of log(RR)
        SE = sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
           = sqrt(1/54 - 1/70 + 1/8 - 1/50)
           = 0.3305
        # The interval is built on the log scale because RR is a ratio: its
        # sampling distribution is skewed, but log(RR) is roughly normal.

Step 5  95% confidence interval, lower limit
        lower = exp(log(RR) - z * SE)
              = exp(1.5731 - 1.96 * 0.3305)
              = 2.5226

Read the derivation top to bottom and notice what it is doing.

**Steps 1 and 2** compute the two risks separately before dividing them. This
matters pedagogically: a student who computes `54/70 = 0.7714` and stops has
done step 1 correctly. Telling them "wrong, the answer is 4.82" hides that
from both of you.

**Step 3** is the ratio. **Steps 4-6** build the confidence interval, and note
that the interval is computed on the **log scale** and then exponentiated. That
is why the interval is not symmetric around the estimate: 4.82 sits closer to
2.52 than to 9.22 on the arithmetic scale, but exactly in the middle on the
log scale. Ratios are multiplicative quantities, and their sampling
distribution is much closer to normal after a log transform.

Now the same table as a difference rather than a ratio:

In [4]:
risk_difference(picnic)

── Risk difference ─────────────────────────────────────────────────────────

Data
                    Ill  Well  Total
  Ate potato salad   54    16     70
  Did not eat         8    42     50
  Total              62    58    120

Step 1  Risk of Ill among the ate potato salad
        R1 = a / (a + b)
           = 54 / (54 + 16)
           = 0.7714

Step 2  Risk of Ill among the did not eat
        R0 = c / (c + d)
           = 8 / (8 + 42)
           = 0.16

Step 3  Risk difference
        RD = R1 - R0
           = 0.7714 - 0.16
           = 0.6114

Step 4  Standard error of RD
        SE = sqrt(R1(1-R1)/(a+b) + R0(1-R0)/(c+d))
           = sqrt(0.7714*0.2286/70 + 0.16*0.84/50)
           = 0.0722
        # No log transform here. A difference can be negative, so it is
        # already on a scale where the normal approximation applies
        # directly.

Step 5  95% confidence interval, lower limit
        lower = RD - z * SE
              = 0.6114 - 1.96 * 0.0722
              = 0.

The risk difference is **0.61** — 61 additional cases per 100 people exposed.

Notice the two measures answer genuinely different questions:

* **Risk ratio (4.82)** — how many *times* more likely are the exposed to fall
  ill? A question about strength of association, useful for asking whether
  something is causal.
* **Risk difference (0.61)** — how many *extra cases* does the exposure
  produce? A question about public health impact, useful for deciding where to
  spend money.

A rare exposure can have a huge risk ratio and a negligible risk difference.
Neither number is more correct; they are answers to different questions, and
the confusion between them is behind a great deal of bad science
communication.

The output also reports the **number needed to expose** — the reciprocal of
the risk difference. Here roughly 2 people had to eat the potato salad to
produce one extra case.

---
## Part 3 — The odds ratio

Odds are not risks. The **odds** of an event are the number of times it
happens divided by the number of times it does not — `a/b`, not `a/(a+b)`.

In [5]:
odds_ratio(picnic)

── Odds ratio ──────────────────────────────────────────────────────────────

Data
                    Ill  Well  Total
  Ate potato salad   54    16     70
  Did not eat         8    42     50
  Total              62    58    120

Step 1  Odds of Ill among the ate potato salad
        odds1 = a / b
              = 54 / 16
              = 3.375

Step 2  Odds of Ill among the did not eat
        odds0 = c / d
              = 8 / 42
              = 0.1905

Step 3  Odds ratio
        OR = odds1 / odds0
           = 3.375 / 0.1905
           = 17.7188
        # Equivalently OR = ad / bc, which is why the odds ratio is the
        # same whether you condition on exposure or on outcome. That
        # symmetry is what makes it usable in case-control studies.

Step 4  Standard error of log(OR), Woolf's method
        SE = sqrt(1/a + 1/b + 1/c + 1/d)
           = sqrt(1/54 + 1/16 + 1/8 + 1/42)
           = 0.4794

Step 5  95% confidence interval, lower limit
        lower = exp(log(OR) - z * S

The odds ratio is **17.72**, while the risk ratio for the very same table was
**4.82**.

That is not an error. **The odds ratio is always further from 1 than the risk
ratio**, and the gap widens as the outcome becomes more common. Here 77% of
the exposed fell ill — an extremely common outcome — so the two measures
diverge dramatically.

### Why does anyone use the odds ratio then?

Because it has a property no other measure has: it is **estimable from a
case-control study**. When you sample on the outcome — deliberately recruiting
cases and controls in whatever ratio is convenient — you destroy your ability
to estimate risk. But the odds ratio is unchanged whether you condition on
exposure or on outcome, which is exactly what the note in the derivation above
is pointing at when it says `OR = ad/bc`.

### The rare disease assumption, demonstrated

Let's fix the risk ratio at exactly 2.0 and vary only how common the outcome
is:

In [6]:
options(epibyhand.verbose = 0)   # results only, no derivations

for (R0 in c(0.001, 0.01, 0.05, 0.10, 0.20, 0.40)) {
  n <- 100000
  c_cell <- R0 * n;      d_cell <- n - c_cell
  a_cell <- 2 * R0 * n;  b_cell <- n - a_cell
  tab <- epi2x2(a_cell, b_cell, c_cell, d_cell)
  cat(sprintf("baseline risk %5.3f   RR = %.3f   OR = %.4f\n",
              R0, estimate(risk_ratio(tab)), estimate(odds_ratio(tab))))
}

options(epibyhand.verbose = 2)   # back to full derivations

baseline risk 0.001   RR = 2.000   OR = 2.0020
baseline risk 0.010   RR = 2.000   OR = 2.0204
baseline risk 0.050   RR = 2.000   OR = 2.1111
baseline risk 0.100   RR = 2.000   OR = 2.2500
baseline risk 0.200   RR = 2.000   OR = 2.6667
baseline risk 0.400   RR = 2.000   OR = 6.0000


The risk ratio is pinned at 2.000 in every row. The odds ratio drifts from
**2.002** to **6.000**.

This is the rare disease assumption made concrete. When the outcome is rare
(say under 5-10%), the odds ratio is a usable approximation to the risk ratio,
which is why case-control studies of rare diseases can report an OR and have
epidemiologists read it as a risk ratio. When the outcome is common, treating
an odds ratio as though it were a risk ratio **badly overstates the effect** —
and this happens constantly in published abstracts.

---
## Part 4 — Controlling the detail

Full derivations are the point of the package, but you do not always want
them. Two global options control the output.

In [7]:
options(epibyhand.verbose = 0)   # the answer alone
odds_ratio(picnic)

── Odds ratio ──────────────────────────────────────────────────────────────

Result
  OR = 17.7188 (95% CI 6.9241 to 45.3422)

Notes
  These data are tabulated as a cohort, where risk is estimable. The risk
  ratio here is 4.8214 against an odds ratio of 17.7188 -- the odds ratio
  is the more extreme of the two, and always will be. The two converge only
  when the outcome is rare; baseline risk here is 0.16.


In [8]:
options(epibyhand.verbose = 1)   # add the symbolic formulas, drop the numbers
odds_ratio(picnic)

── Odds ratio ──────────────────────────────────────────────────────────────

Step 1  Odds of Ill among the ate potato salad
        odds1 = a / b
              = 3.375

Step 2  Odds of Ill among the did not eat
        odds0 = c / d
              = 0.1905

Step 3  Odds ratio
        OR = odds1 / odds0
           = 17.7188
        # Equivalently OR = ad / bc, which is why the odds ratio is the
        # same whether you condition on exposure or on outcome. That
        # symmetry is what makes it usable in case-control studies.

Step 4  Standard error of log(OR), Woolf's method
        SE = sqrt(1/a + 1/b + 1/c + 1/d)
           = 0.4794

Step 5  95% confidence interval, lower limit
        lower = exp(log(OR) - z * SE)
              = 6.9241

Step 6  95% confidence interval, upper limit
        upper = exp(log(OR) + z * SE)
              = 45.3422


Result
  OR = 17.7188 (95% CI 6.9241 to 45.3422)

Notes
  These data are tabulated as a cohort, where risk is estimable. The risk
  ratio

In [9]:
options(epibyhand.verbose = 2)   # full worked solution (the default)
options(epibyhand.digits  = 3)   # and round to 3 significant digits
odds_ratio(picnic)

── Odds ratio ──────────────────────────────────────────────────────────────

Data
                    Ill  Well  Total
  Ate potato salad   54    16     70
  Did not eat         8    42     50
  Total              62    58    120

Step 1  Odds of Ill among the ate potato salad
        odds1 = a / b
              = 54 / 16
              = 3.375

Step 2  Odds of Ill among the did not eat
        odds0 = c / d
              = 8 / 42
              = 0.19

Step 3  Odds ratio
        OR = odds1 / odds0
           = 3.375 / 0.19
           = 17.719
        # Equivalently OR = ad / bc, which is why the odds ratio is the
        # same whether you condition on exposure or on outcome. That
        # symmetry is what makes it usable in case-control studies.

Step 4  Standard error of log(OR), Woolf's method
        SE = sqrt(1/a + 1/b + 1/c + 1/d)
           = sqrt(1/54 + 1/16 + 1/8 + 1/42)
           = 0.479

Step 5  95% confidence interval, lower limit
        lower = exp(log(OR) - z * SE)
   

In [10]:
options(epibyhand.digits = 4)    # restore the default before continuing

A natural teaching pattern: `verbose = 2` when introducing a measure for the
first time, `verbose = 1` when the class already knows the formula and you are
reminding them, `verbose = 0` once you are just doing analysis.

---
## Part 5 — `check_work()`

This is the feature no other package has. Give it a value and it compares
against the final answer. When that does not match, it searches **every
intermediate step** for one that does.

In [11]:
d <- odds_ratio(picnic)

check_work(d, 17.72)

Correct. 17.72 matches the final estimate (OR).


In [12]:
check_work(d, 3.375)

Not a match. You gave 3.375; the final estimate (OR) is 17.7188.

Your value does match step 1: Odds of Ill among the ate potato salad.
  odds1 = a / b
You may have stopped early. The next step is: Odds of Ill among the did not eat.


That second student did not fail. They computed the odds among the exposed
(`54/16 = 3.375`) and stopped — which is a completely different problem from
an arithmetic slip, and needs a different sentence from the person teaching
them.

Compare a genuine arithmetic error:

In [13]:
check_work(d, 9.99)

Not a match. You gave 9.99; the final estimate (OR) is 17.7188.

It does not match any intermediate step either, so the slip is probably
arithmetic rather than a wrong stopping point. Print the derivation to
compare line by line.


No intermediate step matches, so the tool says so and tells you to compare the
derivation line by line.

You can also check one specific step by name, and loosen or tighten the
tolerance:

In [14]:
check_work(d, 0.1905, step = "odds0")    # odds among the unexposed: 8/42

Correct. 0.1905 matches step 2 (Odds of Ill among the did not eat).


---
## Part 6 — Attributable fractions

An attributable fraction asks: **what proportion of disease would disappear if
we removed the exposure?** There are two versions and confusing them is a
classic exam mistake.

In [15]:
attributable_fraction(picnic, among = "exposed")

── Attributable fraction among the exposed ─────────────────────────────────

Data
                    Ill  Well  Total
  Ate potato salad   54    16     70
  Did not eat         8    42     50
  Total              62    58    120

Step 1  Risk of Ill among the ate potato salad
        R1 = a / (a + b)
           = 54 / (54 + 16)
           = 0.7714

Step 2  Risk of Ill among the did not eat
        R0 = c / (c + d)
           = 8 / (8 + 42)
           = 0.16

Step 3  Risk ratio
        RR = R1 / R0
           = 0.7714 / 0.16
           = 4.8214

Step 4  Attributable fraction among the exposed
        AFe = (R1 - R0) / R1
            = (0.7714 - 0.16) / 0.7714
            = 0.7926

Step 5  The same quantity from the risk ratio alone
        AFe = (RR - 1) / RR
            = (4.8214 - 1) / 4.8214
            = 0.7926
        # Identical, because dividing through by R0 cancels it. This is why
        # the attributable fraction among the exposed can be computed from
        # a case-cont

**AFe = 0.79.** Among people who ate the potato salad, 79% of their illness is
attributable to it. The remaining 21% is the background rate — they would have
been ill anyway.

Note the second step: the same quantity comes out of `(RR - 1)/RR` using only
the risk ratio. That is why AFe can be computed from a case-control study,
where absolute risks are unavailable but the ratio is estimable.

Now the population version:

In [16]:
attributable_fraction(picnic, among = "population")

── Population attributable fraction ────────────────────────────────────────

Data
                    Ill  Well  Total
  Ate potato salad   54    16     70
  Did not eat         8    42     50
  Total              62    58    120

Step 1  Risk of Ill among the ate potato salad
        R1 = a / (a + b)
           = 54 / (54 + 16)
           = 0.7714

Step 2  Risk of Ill among the did not eat
        R0 = c / (c + d)
           = 8 / (8 + 42)
           = 0.16

Step 3  Risk ratio
        RR = R1 / R0
           = 0.7714 / 0.16
           = 4.8214

Step 4  Risk in the whole population
        Rt = (a + c) / n
           = (54 + 8) / 120
           = 0.5167

Step 5  Population attributable fraction, directly from risks
        PAF = (Rt - R0) / Rt
            = (0.5167 - 0.16) / 0.5167
            = 0.6903
        # The share of the population's risk that would disappear if
        # everyone had the risk of the unexposed.

Step 6  Levin's formula, from exposure prevalence
        PAF = p

**PAF = 0.69**, lower than the AFe of 0.79.

Why? Because only 58% of attendees ate the potato salad. The unexposed 42%
contribute their cases to the denominator but have nothing attributable to
remove, which dilutes the fraction.

### The three formulas

The derivation computes the PAF **three ways** — directly from risks, by
Levin's formula from exposure prevalence, and by Miettinen's formula from the
proportion of cases exposed — and they agree to the last digit.

Textbooks present these as though they were alternatives to choose between.
They are not. They are **one quantity written three ways**, and which one you
use depends only on which inputs you happen to have:

* **Direct** — you have the full table.
* **Levin** — you have a risk ratio from one study and exposure prevalence
  from another.
* **Miettinen** — you have a case-control study and know what fraction of
  cases were exposed.

### The warning that matters

PAF depends on how common the exposure is, so **it does not transfer between
populations** the way a risk ratio does. A strong risk factor that is rare has
a small PAF; a weak one that is universal can have a large one. Quoting a PAF
from one country's study as though it applied to another is a real and common
error.

---
## Part 7 — Confounding and stratified analysis

### The Whickham data

In 1972-74 a survey in Whickham, England recorded whether each participant
smoked. Twenty years later the survivors were identified. What follows is the
1314 women in that cohort — a standard illustration of Simpson's paradox
(Appleton, French and Vanderpump, 1996, *The American Statistician* **50**,
340-341).

Start where a student would: smoking and death, ignoring everything else.

In [17]:
whickham_crude <- epi2x2(139, 443, 230, 502,
                         exposure = c("Smoker", "Non-smoker"),
                         outcome  = c("Dead", "Alive"))

options(epibyhand.verbose = 1)
odds_ratio(whickham_crude)
risk_ratio(whickham_crude)

── Odds ratio ──────────────────────────────────────────────────────────────

Step 1  Odds of Dead among the smoker
        odds1 = a / b
              = 0.3138

Step 2  Odds of Dead among the non-smoker
        odds0 = c / d
              = 0.4582

Step 3  Odds ratio
        OR = odds1 / odds0
           = 0.6848
        # Equivalently OR = ad / bc, which is why the odds ratio is the
        # same whether you condition on exposure or on outcome. That
        # symmetry is what makes it usable in case-control studies.

Step 4  Standard error of log(OR), Woolf's method
        SE = sqrt(1/a + 1/b + 1/c + 1/d)
           = 0.1257

Step 5  95% confidence interval, lower limit
        lower = exp(log(OR) - z * SE)
              = 0.5353

Step 6  95% confidence interval, upper limit
        upper = exp(log(OR) + z * SE)
              = 0.8761


Result
  OR = 0.6848 (95% CI 0.5353 to 0.8761)

Notes
  These data are tabulated as a cohort, where risk is estimable. The risk
  ratio here is 0.7

**The odds ratio is 0.68 and the confidence interval excludes 1.**

Read naively: smoking is protective, and significantly so. The risk ratio
agrees. A report written at this point would be internally consistent,
statistically significant, and **false**.

### What went wrong

Smoking was less common among the oldest women in the survey, and the oldest
women were the ones most likely to die within twenty years. Age is associated
with the exposure *and* independently predicts the outcome. That is the
definition of a **confounder**.

So look inside age groups, where age is held fixed by construction:

In [18]:
whickham <- epi_strata(
  c(15, 270,  12, 327),   # 18-44:  smoker dead/alive, non-smoker dead/alive
  c(80, 167,  53, 147),   # 45-64
  c(44,   6, 165,  28),   # 65+
  labels   = c("18-44", "45-64", "65+"),
  exposure = c("Smoker", "Non-smoker"),
  outcome  = c("Dead", "Alive")
)

whickham

18-44
              Dead  Alive  Total
  Smoker        15    270    285
  Non-smoker    12    327    339
  Total         27    597    624

45-64
              Dead  Alive  Total
  Smoker        80    167    247
  Non-smoker    53    147    200
  Total        133    314    447

65+
              Dead  Alive  Total
  Smoker        44      6     50
  Non-smoker   165     28    193
  Total        209     34    243


In [19]:
round(mh_odds_ratio(whickham)$stratum_estimates, 3)

18-44 45-64   65+ 
1.514 1.329 1.244 


**All three age groups give an odds ratio above 1.** The crude estimate was
0.68. Adjustment here does not merely shift the estimate — it **reverses** it.

Now pool them:

In [20]:
options(epibyhand.verbose = 2)
mh_odds_ratio(whickham)

── Mantel-Haenszel odds ratio ──────────────────────────────────────────────

Data
  18-44
                Dead  Alive  Total
    Smoker        15    270    285
    Non-smoker    12    327    339
    Total         27    597    624

  45-64
                Dead  Alive  Total
    Smoker        80    167    247
    Non-smoker    53    147    200
    Total        133    314    447

  65+
                Dead  Alive  Total
    Smoker        44      6     50
    Non-smoker   165     28    193
    Total        209     34    243

Step 1  Odds ratio within each stratum
        OR_i = (a_i * d_i) / (b_i * c_i)

        Stratum   a    b    c    d  n_i    OR_i
        -------  --  ---  ---  ---  ---  ------
          18-44  15  270   12  327  624  1.5139
          45-64  80  167   53  147  447  1.3287
            65+  44    6  165   28  243  1.2444

        # Look at these before pooling. If they disagree substantially the
        # stratifying variable is an effect modifier, and a single pooled
 

### The weights are the point

Almost no software shows you this. `S_i` is what each stratum contributes, and
the pooled estimate is a **weighted average of the stratum odds ratios** with
weights `S_i`.

That has a consequence you can check by eye: **`OR_MH` must fall between the
smallest and largest stratum estimate.** Here 1.350 sits between 1.244 and
1.514. It does. If yours does not, your arithmetic is wrong.

The crude estimate of 0.68 does not fall in that range, and could not, because
it is **not an average of these numbers at all**. It is a different quantity
that happens to be computed from the same table.

The last step of the derivation prints the crude estimate beside the adjusted
one and reports the percentage change, so confounding is arithmetic rather
than assertion. The usual working rule is that a change beyond about 10%
indicates confounding — but note that this is a **judgement about the data,
not a hypothesis test**. Do not decide it with a p-value.

### The adjusted interval crosses 1

`OR_MH = 1.350, 95% CI 0.961 to 1.896`. The crude interval excluded 1; the
adjusted one does not.

This is worth sitting with. Adjustment is about getting the **right** answer,
not a bigger or more significant one. Here the honest conclusion from these
1314 women is that smoking is associated with higher 20-year mortality, with
an effect estimate compatible with anything from a trivial protective effect
to nearly a doubling. The confidently significant protective effect was an
artifact.

---
## Part 8 — Was pooling legitimate?

A single pooled odds ratio only means something if **one odds ratio underlies
every stratum**. If the strata genuinely differ, the stratifying variable is
an **effect modifier**, and pooling destroys the finding rather than reporting
it.

The Breslow-Day test asks whether the observed spread is more than chance.

In [21]:
options(epibyhand.verbose = 1)
homogeneity(whickham)

── Breslow-Day test of homogeneity ─────────────────────────────────────────

Step 1  Expected exposed cases in each stratum under a common OR
        A_i = root of  (1-psi)A^2 + (N - n1 - m1 + psi(n1+m1))A - psi*n1*m1 = 0
        # A_i is what cell a would be if this stratum had exactly the
        # pooled odds ratio, holding its margins fixed. Solving a quadratic
        # is the one step here you would not do by hand.

Step 2  Contribution of each stratum to the statistic
        X2_i = (a_i - A_i)^2 / Var(A_i)
        # A single large contribution means one stratum is driving the
        # result.

Step 3  Statistic, with Tarone's correction
        X2 = sum(X2_i) - (sum(a_i) - sum(A_i))^2 / sum(Var(A_i))
           = 0.1182

Step 4  Reference distribution
        p = P(chi-squared with K - 1 df > X2)
          = 0.9426


Result
  X2 = 0.1182

Notes
  p = 0.9426 on 2 degrees of freedom.
  A large p-value is not evidence that the odds ratios are equal. This test
  has poor power wi

X-squared = 0.118 on 2 degrees of freedom, p = 0.94. The stratum estimates
(1.51, 1.33, 1.24) are about as homogeneous as random variation allows, and no
single stratum strains against the others. Pooling is comfortable here.

### The caveat the function prints anyway

**A large p-value is not evidence that the odds ratios are equal.** This test
has poor power, especially with small strata, so it will often fail to reject
whether or not effect modification is present. The stratum-specific estimates
you inspected before pooling remain the more informative thing.

### What effect modification actually looks like

Contrast a fabricated study where an exposure is harmful in younger people and
null in older ones:

In [22]:
em <- epi_strata(
  c(60, 40, 30, 70),      # under 50:     OR = (60*70)/(40*30) = 3.5
  c(50, 50, 50, 50),      # 50 and over:  OR = (50*50)/(50*50) = 1.0
  labels = c("Under 50", "50 and over")
)

round(mh_odds_ratio(em)$stratum_estimates, 3)
homogeneity(em)

   Under 50 50 and over 
        3.5         1.0 
── Breslow-Day test of homogeneity ─────────────────────────────────────────

Step 1  Expected exposed cases in each stratum under a common OR
        A_i = root of  (1-psi)A^2 + (N - n1 - m1 + psi(n1+m1))A - psi*n1*m1 = 0
        # A_i is what cell a would be if this stratum had exactly the
        # pooled odds ratio, holding its margins fixed. Solving a quadratic
        # is the one step here you would not do by hand.

Step 2  Contribution of each stratum to the statistic
        X2_i = (a_i - A_i)^2 / Var(A_i)
        # A single large contribution means one stratum is driving the
        # result.

Step 3  Statistic, with Tarone's correction
        X2 = sum(X2_i) - (sum(a_i) - sum(A_i))^2 / sum(Var(A_i))
           = 9.3447

Step 4  Reference distribution
        p = P(chi-squared with K - 1 df > X2)
          = 0.0022


Result
  X2 = 9.3447

Notes
  p = 0.0022 on 1 degrees of freedom.
  A large p-value is not evidence that the od

X-squared = 9.34, p = 0.002. The strata disagree.

The Mantel-Haenszel estimate for these data is **1.81** — a number that
describes neither group. It is the average of a real effect and no effect, and
reporting it alone would hide the actual finding, which is that **the exposure
matters for younger people and not for older ones**.

**Confounding and effect modification are different things and call for
different responses:**

| | What it is | What to do |
|---|---|---|
| **Confounding** | A nuisance distorting the crude estimate | Adjust it away, report the pooled estimate |
| **Effect modification** | A real feature of how the world works | Report the strata separately — it *is* the finding |

---
## Part 9 — Building problem sets

`steps_table()` returns the whole derivation as a data frame, which is what
you want for generating answer keys or rendering the working somewhere the
package does not reach.

In [23]:
tt <- steps_table(mh_odds_ratio(whickham))
tt[, c("symbol", "label", "result")]

    symbol                                                  label    result
1     OR_i                         Odds ratio within each stratum        NA
2     <NA>                                  Stratum contributions        NA
3    OR_MH                                      Pooled odds ratio 1.3499462
4       SE Standard error of log(OR_MH), Robins-Breslow-Greenland 0.1733784
5    lower                   95% confidence interval, lower limit 0.9610291
6    upper                   95% confidence interval, upper limit 1.8962535
7 OR_crude                  Crude odds ratio, ignoring the strata 0.6848366


Every intermediate quantity is addressable. A few things that follow:

* Generate an answer key by pulling the `result` column.
* Write a grading script that compares a student's submitted intermediate
  values against `tt$result` and reports which step they diverged at — the
  same logic `check_work()` uses.
* Export the `label` and `formula` columns into a worksheet, leaving `result`
  blank for students to fill in.

`estimate()` and `confint()` extract the headline numbers when you want to use
a result programmatically rather than read it:

In [24]:
d <- mh_odds_ratio(whickham)
estimate(d)
confint(d)
confint(d, level = 0.99)

[1] 1.349946
[1] 0.9610291 1.8962535
[1] 0.9610291 1.8962535


---
---
# Exercises

Work each one before opening the answer. The explanations are where most of
the learning is.

## Exercise 1 — Build a table and check your arithmetic

A case-control study of a rare cancer recruits **90 cases** and **120
controls**. Among the cases, **60** report the exposure. Among the controls,
**40** do.

1. Build the 2 x 2 table with `epi2x2()`.
2. Compute the odds ratio *by hand* before running anything.
3. Verify with `check_work()`.
4. Should you also compute a risk ratio? Why or why not?

In [25]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
cc <- epi2x2(60, 40, 30, 80,
             exposure = c("Exposed", "Unexposed"),
             outcome  = c("Case", "Control"))

d <- odds_ratio(cc)
check_work(d, 4.0)
```

**The table.** 90 cases of whom 60 exposed leaves 30 unexposed cases. 120
controls of whom 40 exposed leaves 80 unexposed controls. In a, b, c, d order
that is **60, 40, 30, 80**.

**By hand.** `OR = ad/bc = (60 x 80)/(40 x 30) = 4800/1200 = 4.0`. Equivalently
`odds1/odds0 = (60/40)/(30/80) = 1.5/0.375 = 4.0`.

**No risk ratio.** This is the important part. The package will happily compute
one — it gets 2.2 — but that number is **meaningless**. The investigators chose
to recruit 90 cases and 120 controls; that ratio is an artifact of study design,
not of nature. Recruiting 240 controls instead would change the "risk ratio"
and leave the odds ratio untouched.

This is exactly why `epi2x2()` lets you label the columns `Case`/`Control`.
The labels are the reminder.

</details>

## Exercise 2 — When is an odds ratio a risk ratio?

A trial reports an odds ratio of **2.5** for a side effect.

1. If the side effect occurs in **1%** of the control group, roughly what is
   the risk ratio?
2. If it occurs in **40%** of the control group, roughly what is the risk
   ratio?
3. A journalist writes "people on the drug were 2.5 times as likely to have
   the side effect." When is that fair?

*Hint: `R1 = (OR x R0) / (1 - R0 + OR x R0)`, then `RR = R1/R0`.*

In [26]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
or_to_rr <- function(OR, R0) {
  R1 <- (OR * R0) / (1 - R0 + OR * R0)
  R1 / R0
}

or_to_rr(2.5, 0.01)   # 2.463
or_to_rr(2.5, 0.40)   # 1.786
```

**1. R0 = 1%** → RR ≈ **2.46**. Essentially the same as the odds ratio. You can
verify by building the table: with 100,000 per arm, `epi2x2(2463, 97537, 1000,
99000)` gives OR ≈ 2.5 and RR ≈ 2.46.

**2. R0 = 40%** → RR ≈ **1.79**. The odds ratio overstates the risk ratio by
about 40%.

**3.** The journalist is fair in case 1 and **wrong in case 2**. "2.5 times as
likely" is a statement about risk, and when the outcome is common the odds
ratio is simply not that number.

The general rule: the odds ratio is always further from 1 than the risk ratio,
and they converge only when the outcome is rare in *both* groups. The
convenient threshold is around 10%, but there is nothing magic about it — the
divergence is continuous, as the gradient in Part 3 showed.

This is not a pedantic point. Misreporting odds ratios as risk ratios is one
of the most common statistical errors in published health journalism, and it
systematically exaggerates effect sizes.

</details>

## Exercise 3 — Two different questions

Two exposures in the same population of 10,000 people:

* **Exposure A**: risk 0.20 in the exposed, 0.10 in the unexposed. 50% of the
  population is exposed.
* **Exposure B**: risk 0.50 in the exposed, 0.05 in the unexposed. 1% of the
  population is exposed.

1. Which has the larger risk ratio?
2. Which has the larger population attributable fraction?
3. You run a health department with money for exactly one campaign. Which do
   you target?

In [27]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
options(epibyhand.verbose = 0)

A <- epi2x2(1000, 4000,  500, 4500)   # 5000 exposed, 5000 unexposed
B <- epi2x2(  50,   50,  495, 9405)   # 100 exposed, 9900 unexposed

estimate(risk_ratio(A))                                    # 2
estimate(risk_ratio(B))                                    # 10
estimate(attributable_fraction(A, among = "population"))   # 0.333
estimate(attributable_fraction(B, among = "population"))   # 0.083
```

**1. Exposure B**, by a wide margin: RR = 10 versus RR = 2.

**2. Exposure A**, also by a wide margin: PAF ≈ 33% versus ≈ 8%.

**3. Exposure A** — assuming the campaigns are equally effective and equally
expensive.

The reasoning: B is a much more *dangerous* exposure, but it touches 1% of the
population. A is only moderately dangerous but half the population has it.
Eliminating A prevents about a third of all cases; eliminating B prevents about
one in twelve.

This is the single most useful thing the attributable fraction does. The risk
ratio answers *"is this exposure harmful?"* — a question about biology and
causation. The PAF answers *"how much of our disease burden does it cause
here?"* — a question about this population, and the answer changes when you
cross a border.

A caveat worth stating: this analysis assumes both campaigns would be equally
successful at removing the exposure. If A is deeply embedded in the culture and
B is one contaminated water source you could fix on Tuesday, the calculus
changes. PAF tells you the size of the prize, not the cost of winning it.

</details>

## Exercise 4 — Diagnose the discrepancy

A colleague sends you a stratified analysis and asks what to report.

```
Stratum 1:  a=30, b=70, c=20, d=80
Stratum 2:  a=60, b=40, c=45, d=55
```

1. Compute the crude odds ratio and the Mantel-Haenszel odds ratio.
2. Run the homogeneity test.
3. Is the stratifying variable a confounder, an effect modifier, both, or
   neither? What do you tell your colleague to report?

In [28]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
s <- epi_strata(c(30, 70, 20, 80), c(60, 40, 45, 55),
                labels = c("Stratum 1", "Stratum 2"))

d <- mh_odds_ratio(s)
round(d$stratum_estimates, 3)   # 1.714 and 1.833
d$crude                         # 1.891
estimate(d)                     # 1.780
homogeneity(s)                  # X2 = 0.043, p = 0.84
```

**Neither, to any consequential degree.**

* Stratum estimates **1.71** and **1.83** are close together, and Breslow-Day
  gives p = 0.84. **No effect modification.**
* Crude **1.89** versus adjusted **1.78** is a change of about 6%, below the
  usual 10% working threshold. **No meaningful confounding.**

Tell your colleague either number is defensible, and to report the adjusted
one — 1.78 — because adjusting when you did not need to costs almost nothing,
whereas failing to adjust when you did need to is a real error.

**The wider point.** Stratifying and finding nothing is a perfectly good
result, and it is the most common one. The Whickham example in Part 7 is
memorable precisely because reversals are *rare*. A student who has only ever
seen Simpson's paradox examples will over-read every small difference between
crude and adjusted estimates as meaningful.

Note also that the 10% rule is a convention, not a law. It is a judgement about
whether the shift matters for your substantive question — which is why it is
not, and must not be, a hypothesis test.

</details>

## Exercise 5 — Find the student's error

Three students submit answers for the odds ratio of `epi2x2(36, 14, 30, 25)`.
The correct answer is 2.1429.

* Amara writes **0.72**
* Ben writes **2.5714**
* Chidi writes **1.32**

Use `check_work()` to work out what each did, then say what feedback each
needs.

In [29]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
d <- odds_ratio(epi2x2(36, 14, 30, 25))

check_work(d, 0.72)     # matches nothing in this derivation
check_work(d, 2.5714)   # matches step 1: odds among the exposed
check_work(d, 1.32)     # matches nothing here either
```

**Ben** computed `36/14 = 2.5714`, the odds among the exposed, and stopped.
`check_work()` says so explicitly and names the next step. His feedback is
"you were most of the way there, finish the division" — a one-sentence fix.

**Amara** wrote 0.72, which is `36/50` — the *risk* among the exposed, not the
odds. `check_work()` does not find it, because it is not a step in this
derivation at all. Her error is conceptual: she is dividing by the row total
instead of by the other cell. She needs the definition of odds retaught, not
her arithmetic checked.

**Chidi** wrote 1.32, which is the *risk ratio* for this table. He answered a
different question. His feedback is about reading the question and about when
each measure is appropriate.

**Why this matters.** All three are "wrong" in a gradebook, but they need three
completely different conversations. Ben is 30 seconds from correct; Amara has a
definitional gap; Chidi has a conceptual one. Software that only says "the
answer is 2.1429" flattens all three into the same non-feedback.

Note the limit too: `check_work()` can only find errors that correspond to a
step it actually computed. Amara's and Chidi's errors are invisible to it — the
tool narrows the search, it does not replace a teacher looking at the work.

</details>

## Exercise 6 — Reconstruct a published result

A paper reports: *"Among the 400 exposed workers, 120 developed the condition.
The risk ratio compared with 600 unexposed workers was 2.0."*

1. Reconstruct the full 2 x 2 table.
2. Compute the attributable fraction among the exposed.
3. The paper claims eliminating the exposure would prevent 50% of all cases in
   this workforce. Check it.

In [30]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

```r
# R1 = 120/400 = 0.30.  RR = 2 so R0 = 0.15, giving 90 cases among 600.
tab <- epi2x2(120, 280, 90, 510)

options(epibyhand.verbose = 0)
estimate(attributable_fraction(tab, among = "exposed"))      # 0.5
estimate(attributable_fraction(tab, among = "population"))   # 0.2857
```

**1. The table.** `R1 = 120/400 = 0.30`. Since RR = 2, `R0 = 0.15`, so
`0.15 x 600 = 90` cases among the unexposed. Cells: **120, 280, 90, 510**.

**2. AFe = 0.50.** Half the illness among exposed workers is attributable to
the exposure. Sanity check: `(RR - 1)/RR = 1/2`. Whenever the risk ratio is
exactly 2, the AFe is exactly 50%.

**3. The claim is wrong.** It quotes the AFe but describes the PAF.

The PAF is **0.286**. Eliminating the exposure would prevent about 29% of cases
in this workforce, not 50%, because 60% of the workforce is unexposed and
contributes cases that have nothing to do with the exposure.

This is a real and frequent error in occupational and environmental health
reporting, and it always errs in the same direction — overstating the benefit
of an intervention. The tell is the phrase **"of all cases"**: that is a
population claim, so it needs a population fraction.

</details>

## Exercise 7 — Choose the measure

For each scenario, say which measure you would report and why.

1. A case-control study of a rare congenital defect and a medication taken in
   pregnancy.
2. A vaccine trial where 2% of the placebo arm and 0.5% of the vaccine arm are
   infected.
3. A city deciding whether to fund a smoking cessation programme.
4. A cohort study where 60% of the exposed and 30% of the unexposed develop
   hypertension.

In [31]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

**1. Odds ratio.** In a case-control design it is the only valid measure of
association — risk is not estimable when you sample on the outcome. And because
the defect is rare, the OR approximates the risk ratio, so it can be
*interpreted* as one.

**2. Risk ratio**, or its complement, vaccine efficacy. `RR = 0.005/0.02 =
0.25`, so efficacy is `1 - RR = 75%`. This is a trial, so risks are estimable
and there is no reason to reach for odds. Report the risk difference alongside
it — 1.5 percentage points — because that is what determines how many people
must be vaccinated to prevent one infection.

**3. Population attributable fraction**, plus the risk difference. The city is
not asking "is smoking harmful?" — that is settled. It is asking "how much of
our disease burden would this programme remove?", which is a question about
impact in this population, and PAF and RD are the impact measures.

**4. Risk ratio and risk difference — not the odds ratio.** With 60% and 30%
outcomes the OR would be 3.5 while the RR is 2.0, and almost every reader will
misinterpret the 3.5. When risks are estimable and the outcome is common, the
odds ratio is the wrong choice for communication even though it is not wrong
arithmetically.

**The general principle.** The measure follows from two things: the study
design, which determines what is *estimable*, and the question, which
determines what is *relevant*. Ratios speak to causation; differences and
attributable fractions speak to impact.

</details>

## Exercise 8 — Design your own confounding

Construct a 2 x 2 x 2 stratified dataset from scratch in which:

* the crude odds ratio is **below 1**,
* both stratum-specific odds ratios are **above 1**,
* the Breslow-Day test does **not** reject homogeneity.

Verify with the package. Then explain what makes it work.

In [32]:
# Your code here

<details>
<summary><b>Show answer</b></summary>

One construction:

```r
mine <- epi_strata(
  c( 20, 180,   5,  95),   # low-risk stratum,  mostly unexposed
  c(160,  40,  75,  25),   # high-risk stratum, mostly exposed
  labels = c("Low risk", "High risk")
)

d <- mh_odds_ratio(mine)
round(d$stratum_estimates, 3)   # 2.111 and 1.333
d$crude                         # 0.847
estimate(d)                     # 1.512
homogeneity(mine)               # p is comfortably above 0.05
```

**What makes it work** is two conditions holding at once, which is precisely the
definition of a confounder:

1. **The stratifying variable is associated with the outcome.** Baseline risk
   is about 5% in the low-risk stratum and 75% in the high-risk one.
2. **The stratifying variable is associated with the exposure.** The low-risk
   stratum is 67% unexposed; the high-risk stratum is 67% exposed.

Collapsing the strata pools the mostly-unexposed low-risk people with the
mostly-exposed high-risk people. The unexposed group ends up dominated by
low-risk individuals, so it looks artificially healthy, and the exposure looks
protective by comparison.

The strength of the reversal depends on how extreme both associations are. Make
either one weaker and you get ordinary confounding — a shift without a sign
change — which, as Exercise 4 noted, is the far more common situation in
practice.

**Worth noticing:** this took deliberate effort to construct. Full reversals
need both associations to be strong and to point in cooperating directions.
That is why Whickham is famous. If you had to work this hard to build one, you
should be correspondingly sceptical when you think you have found one in real
data — check your table orientation first.

</details>

---
---
## Where to go next

* **Package documentation** — `help(package = "epibyhand")`
* **The vignette** — `vignette("epibyhand")`, which works the Whickham data
  through as a single continuous narrative
* **CRAN** — https://cran.r-project.org/package=epibyhand
* **Source and issues** — https://github.com/rajsubediresearch/epibyhand

### Reference

The Whickham data are from Appleton, D. R., French, J. M. and Vanderpump,
M. P. J. (1996). Ignoring a covariate: an example of Simpson's paradox.
*The American Statistician* **50**(4), 340-341.

### Citing

```r
citation("epibyhand")
```